In [1]:
import numpy as np
from parameters import get_parameters, get_slider_params, calculate_derived_parameters
from model_run import run_model_dash
from global_func import reset_flags, reset_E, reset_HSS, reset_S
import math
import matplotlib.pyplot as plt
import pandas as pd
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

In [ ]:
total_runs = 50
seeds = np.random.default_rng(2025).integers(low=0, high=1e6, size=total_runs)
MODEL = {
    "int_period": 36,
    "n_months": 36,
}
slider_params = get_slider_params()
results = []
for run in range(total_runs):
    base_seed = seeds[run]
    rng_param = np.random.default_rng(base_seed)
    b_param = get_parameters(rng = rng_param)
    b_param = calculate_derived_parameters(b_param)
    b_flags = reset_flags()
    b_HSS = reset_HSS(slider_params)
    b_S = reset_S(slider_params)
    b_E = reset_E()
    b_param.update({"E": b_E, "S": b_S, "HSS": b_HSS
    })
    n_months = MODEL["n_months"]
    int_period = MODEL["int_period"]
    _, outcomes = run_model_dash(b_param, b_flags, n_months, int_period, base_seed=base_seed)

    outcomes['i_CS'] = np.where(outcomes['i_mod'].isin(["EmCS", "ELCS"]), 1, 0)
    outcomes['i_loc_grouped'] = np.where(outcomes['i_loc_new_v2'] == 0, 0,
                                         np.where(outcomes['i_loc_new_v2'] == 1, 1, 2))
    outcomes['i_facility'] = np.where(outcomes['i_loc_new_v2'] > 0, 1, 0)
    outcomes['i_home'] = np.where(outcomes['i_loc_new_v2'] == 0, 1, 0)
    mcr_l23_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 1]['i_comp_death_new'].mean(), nan=0.0)
    mcr_l45_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 2]['i_comp_death_new'].mean(), nan=0.0)
    p_mcr_l23_l45 = mcr_l23_mean / mcr_l45_mean if mcr_l45_mean != 0 else 0

    results.append({
        "p_elcs_fac": outcomes.groupby('i_facility')['i_elec_CS'].mean().get(1, 0),
        "p_elcs_preterm_l45": outcomes[(outcomes['i_loc_grouped'] == 2) & (outcomes['i_preterm'] == 1)]['i_elec_CS'].mean(),
        "p_preterm_l23": outcomes[(outcomes['i_loc_grouped'] == 1)]['i_preterm'].mean(),
        "p_preterm_l45": outcomes[(outcomes['i_loc_grouped'] == 2)]['i_preterm'].mean(),
        "p_cs_l23": outcomes.groupby('i_loc_grouped')['i_CS'].mean().get(1, 0),

        "p_cs_l45": outcomes.groupby('i_loc_grouped')['i_CS'].mean().get(2, 0),
        "p_unnecessary_cs": outcomes[(outcomes['i_mod'] == "EmCS")]['i_unnecessary_cs'].mean(),
        "p_l23_post": (outcomes['i_loc_grouped'] == 1).mean(),
        "p_l45_post": (outcomes['i_loc_grouped'] == 2).mean(),
        "p_mcr_l23_l45": p_mcr_l23_l45,

        "p_SMO_fac": outcomes[(outcomes['i_severe_new'] == 1) & (outcomes['i_facility'] == 1)]['i_mat_death'].mean(),
        "p_pph_l45": outcomes.groupby('i_loc_grouped')['i_pph_new'].mean().get(2, 0),
        "p_mat_sepsis_l45": outcomes.groupby('i_loc_grouped')['i_mat_sepsis'].mean().get(2, 0),
        "p_eclampsia_fac": outcomes.groupby('i_facility')['i_eclampsia'].mean().get(1, 0),
        "p_obstructed_fac": outcomes.groupby('i_facility')['i_OL_final'].mean().get(1, 0),
    })

    # Convert to DataFrame and compute mean
df_results = pd.DataFrame(results).mean().to_dict()
#round 2 digits for each cell of df_results
df_results = {k: round(v, 3) for k, v in df_results.items()}
df_results['p_SMO_fac']

In [ ]:
def find_min_runs(target_tolerance=0.01, max_runs=100, min_runs=10):
    """
    Determine the minimum number of runs needed for stable results across all target metrics.

    Args:
        target_tolerance: Maximum allowed relative standard error (e.g., 0.01 = 1%)
        max_runs: Maximum number of runs to test before giving up

    Returns:
        min_runs: Recommended minimum number of runs
        convergence_data: Dictionary tracking all metrics' convergence
    """
    # Initialize storage for convergence tracking
    target_values = {
    "p_elcs_fac": 0.066,
    "p_elcs_preterm_l45": 0.265,
    "p_preterm_l23": 0.035,
    "p_preterm_l45": 0.209,
    "p_cs_l23": 0.024,

    "p_cs_l45": 0.264,
    "p_unnecessary_cs": 0.67,
    "p_l23_post": 0.27,
    "p_l45_post": 0.38,
    "p_mcr_l23_l45": 0.2,

    #"p_SMO_fac": 0.0375,
    "p_pph_l45": 0.086,
    "p_mat_sepsis_l45": 0.15,
    "p_eclampsia_fac": 0.02,
    "p_obstructed_fac": 0.045,
    }

    convergence_data = {k: {'values': [], 'means': [], 'stderrs': []}
                       for k in target_values.keys()}

    seeds = np.random.default_rng(2025).integers(low=0, high=1e6, size=max_runs)
    MODEL = {"int_period": 36, "n_months": 36}
    slider_params = get_slider_params()

    for run in range(max_runs):
        # Run model (using your existing code structure)
        base_seed = seeds[run]
        rng_param = np.random.default_rng(base_seed)
        b_param = get_parameters(rng=rng_param)
        b_param = calculate_derived_parameters(b_param)
        b_flags = reset_flags()
        b_HSS = reset_HSS(slider_params)
        b_S = reset_S(slider_params)
        b_E = reset_E()
        b_param.update({"E": b_E, "S": b_S, "HSS": b_HSS})

        _, outcomes = run_model_dash(b_param, b_flags, MODEL["n_months"], MODEL["int_period"], base_seed=base_seed)

        # Calculate outcomes (your existing processing code)
        outcomes['i_CS'] = np.where(outcomes['i_mod'].isin(["EmCS", "ELCS"]), 1, 0)
        outcomes['i_loc_grouped'] = np.where(outcomes['i_loc_new_v2'] == 0, 0,
                                           np.where(outcomes['i_loc_new_v2'] == 1, 1, 2))
        outcomes['i_facility'] = np.where(outcomes['i_loc_new_v2'] > 0, 1, 0)
        mcr_l23_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 1]['i_comp_death_new'].mean(), nan=0.0)
        mcr_l45_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 2]['i_comp_death_new'].mean(), nan=0.0)
        p_mcr_l23_l45 = mcr_l23_mean / mcr_l45_mean if mcr_l45_mean != 0 else 0

        # Store results for all target metrics
        run_results = {
        "p_elcs_fac": outcomes.groupby('i_facility')['i_elec_CS'].mean().get(1, 0),
        "p_elcs_preterm_l45": outcomes[(outcomes['i_loc_grouped'] == 2) & (outcomes['i_preterm'] == 1)]['i_elec_CS'].mean(),
        "p_preterm_l23": outcomes[(outcomes['i_loc_grouped'] == 1)]['i_preterm'].mean(),
        "p_preterm_l45": outcomes[(outcomes['i_loc_grouped'] == 2)]['i_preterm'].mean(),
        "p_cs_l23": outcomes.groupby('i_loc_grouped')['i_CS'].mean().get(1, 0),

        "p_cs_l45": outcomes.groupby('i_loc_grouped')['i_CS'].mean().get(2, 0),
        "p_unnecessary_cs": outcomes[(outcomes['i_mod'] == "EmCS")]['i_unnecessary_cs'].mean(),
        "p_l23_post": (outcomes['i_loc_grouped'] == 1).mean(),
        "p_l45_post": (outcomes['i_loc_grouped'] == 2).mean(),
        "p_mcr_l23_l45": p_mcr_l23_l45,

        #"p_SMO_fac": outcomes[(outcomes['i_severe_new'] == 1) & (outcomes['i_facility'] == 1)]['i_mat_death'].mean(),
        "p_pph_l45": outcomes.groupby('i_loc_grouped')['i_pph_new'].mean().get(2, 0),
        "p_mat_sepsis_l45": outcomes.groupby('i_loc_grouped')['i_mat_sepsis'].mean().get(2, 0),
        "p_eclampsia_fac": outcomes.groupby('i_facility')['i_eclampsia'].mean().get(1, 0),
        "p_obstructed_fac": outcomes.groupby('i_facility')['i_OL_final'].mean().get(1, 0),
        }

        # Update convergence tracking
        # Update convergence tracking
        all_metrics_converged = (run + 1 >= min_runs)  # +1 because run starts at 0
        failed_metrics = []

        for metric in target_values:
            value = run_results[metric]
            convergence_data[metric]['values'].append(value)
            n = len(convergence_data[metric]['values'])

            # Handle cases where all values are identical (std=0)
            if np.std(convergence_data[metric]['values']) == 0:
                current_stderr = 0.0
            else:
                current_stderr = np.std(convergence_data[metric]['values'], ddof=1) / np.sqrt(n)

            convergence_data[metric]['means'].append(np.mean(convergence_data[metric]['values']))
            convergence_data[metric]['stderrs'].append(current_stderr)
            # Check convergence
            if target_values[metric] != 0:
                rel_stderr = abs(current_stderr / target_values[metric])
            else:
                rel_stderr = current_stderr  # Absolute error if target=0

            if rel_stderr > target_tolerance:
                all_metrics_converged = False
                failed_metrics.append(metric)

        if all_metrics_converged:
            print(f"\nConvergence achieved at {run+1} runs (all metrics ≤ {target_tolerance*100:.0f}% RSE)")
            return run + 1, convergence_data
        elif run == max_runs - 1:
            print(f"\nWarning: Target tolerance not reached after {max_runs} runs.")
            print(f"Metrics still failing: {failed_metrics[-5:]}...")  # Show last 5 for brevity
            return max_runs, convergence_data

min_runs, conv_data = find_min_runs(target_tolerance=0.05, max_runs=100, min_runs = 10)  # 5% tolerance

#Convergence achieved at 18 runs (all metrics ≤ 5% RSE)

In [ ]:
def find_min_runs(target_tolerance=0.01, max_runs=100, min_runs=10):
    """
    Determine the minimum number of runs needed for stable results across all target metrics.

    Args:
        target_tolerance: Maximum allowed relative standard error (e.g., 0.01 = 1%)
        max_runs: Maximum number of runs to test before giving up

    Returns:
        min_runs: Recommended minimum number of runs
        convergence_data: Dictionary tracking all metrics' convergence
    """
    # Initialize storage for convergence tracking
    target_values = {
    "p_SMO_fac": 0.0375,
    }

    convergence_data = {k: {'values': [], 'means': [], 'stderrs': []}
                       for k in target_values.keys()}

    seeds = np.random.default_rng(2025).integers(low=0, high=1e6, size=max_runs)
    MODEL = {"int_period": 36, "n_months": 36}
    slider_params = get_slider_params()

    for run in range(max_runs):
        # Run model (using your existing code structure)
        base_seed = seeds[run]
        rng_param = np.random.default_rng(base_seed)
        b_param = get_parameters(rng=rng_param)
        b_param = calculate_derived_parameters(b_param)
        b_flags = reset_flags()
        b_HSS = reset_HSS(slider_params)
        b_S = reset_S(slider_params)
        b_E = reset_E()
        b_param.update({"E": b_E, "S": b_S, "HSS": b_HSS})

        _, outcomes = run_model_dash(b_param, b_flags, MODEL["n_months"], MODEL["int_period"], base_seed=base_seed)

        # Calculate outcomes (your existing processing code)
        outcomes['i_CS'] = np.where(outcomes['i_mod'].isin(["EmCS", "ELCS"]), 1, 0)
        outcomes['i_loc_grouped'] = np.where(outcomes['i_loc_new_v2'] == 0, 0,
                                           np.where(outcomes['i_loc_new_v2'] == 1, 1, 2))
        outcomes['i_facility'] = np.where(outcomes['i_loc_new_v2'] > 0, 1, 0)
        mcr_l23_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 1]['i_comp_death_new'].mean(), nan=0.0)
        mcr_l45_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 2]['i_comp_death_new'].mean(), nan=0.0)
        p_mcr_l23_l45 = mcr_l23_mean / mcr_l45_mean if mcr_l45_mean != 0 else 0

        # Store results for all target metrics
        run_results = {
        "p_SMO_fac": outcomes[(outcomes['i_severe_new'] == 1) & (outcomes['i_facility'] == 1)]['i_mat_death'].mean(),
        }

        # Update convergence tracking
        all_metrics_converged = (run + 1 >= min_runs)  # +1 because run starts at 0
        failed_metrics = []

        for metric in target_values:
            value = run_results[metric]
            convergence_data[metric]['values'].append(value)
            n = len(convergence_data[metric]['values'])

            # Handle cases where all values are identical (std=0)
            if np.std(convergence_data[metric]['values']) == 0:
                current_stderr = 0.0
            else:
                current_stderr = np.std(convergence_data[metric]['values'], ddof=1) / np.sqrt(n)

            convergence_data[metric]['means'].append(np.mean(convergence_data[metric]['values']))
            convergence_data[metric]['stderrs'].append(current_stderr)
            # Check convergence
            if target_values[metric] != 0:
                rel_stderr = abs(current_stderr / target_values[metric])
            else:
                rel_stderr = current_stderr  # Absolute error if target=0

            if rel_stderr > target_tolerance:
                all_metrics_converged = False
                failed_metrics.append(metric)

        if all_metrics_converged:
            print(f"\nConvergence achieved at {run+1} runs (all metrics ≤ {target_tolerance*100:.0f}% RSE)")
            return run + 1, convergence_data
        elif run == max_runs - 1:
            print(f"\nWarning: Target tolerance not reached after {max_runs} runs.")
            print(f"Metrics still failing: {failed_metrics[-5:]}...")  # Show last 5 for brevity
            return max_runs, convergence_data

min_runs, conv_data = find_min_runs(target_tolerance=0.10, max_runs=100, min_runs = 10)  # 5% tolerance

#Convergence achieved at 25 runs (all metrics ≤ 10% RSE)

To sum up, I use 30 runs for model calibration in this phase.

In [2]:
# Define the parameter space for Bayesian Optimization
param_space = [
    Real(0.01, 0.2,      name='p_elec_CS|highrisk'), #control "p_elcs_fac"
    Real(0.5, 0.8,      name='p_elec_CS|preterm'),  #control "p_elcs_preterm_l45"
    Real(80, 100,       name='t_l23_l45_preterm'), #control 'p_preterm_l23' and 'p_preterm_l45'
    Real(0.01, 0.10,    name='p_cs_capacity_l23'), #control "p_cs_l23"
    Real(0.10, 0.20,    name='p_cs_capacity_l45'), #control "p_cs_l45"

    Real(0.5, 0.74,     name='sen_comp_trad'),    #control "p_unnecessary_cs"
    Real(0.5, 0.77,     name='spec_comp_trad'),   #control "p_unnecessary_cs"
    Real(50, 90,        name='t_l23_l45_notsevere'), #control 'p_l23_post' and 'p_l45_post' and "p_mcr_l23_l45"
    #Real(50, 100,       name='t_l23_l45_severe'),   #control 'p_l23_post' and 'p_l45_post' and "p_mcr_l23_l45"
    Real(1, 40,         name='delta_severe_vs_notsevere'),  # Initial delta (1 ≤ delta ≤ 45)
    Real(0.01, 0.10,    name='p_comp_severe_lowrisk'), #control "p_SMO_fac"

    Real(0.01, 0.03,    name='p_eclampsia'),
    Real(0.70, 1.00,    name='p_OL_scale'),
    Real(0.01, 0.02,    name='p_pph_other'),
    Real(0.01, 0.05,    name='p_mat_sepsis_other'),
]

def weighted_rmse(df_results, target_weights):
    weighted_errors = []

    for metric, (target, accepted_pct) in target_weights.items():
        simulated = df_results.get(metric, 0)
        error = (simulated - target)
        # Calculate absolute accepted error margin
        accepted_abs_error = target * (accepted_pct / 100)
        # Normalize error by accepted margin (0.5 = half of allowed error)
        normalized_error = error / accepted_abs_error
        weighted_errors.append(normalized_error**2)
    return math.sqrt(sum(weighted_errors) / len(weighted_errors))

# Calibration target values
# target_values = {
#     "p_elcs_fac": 0.066,         #5% accepted error
#     "p_elcs_preterm_l45": 0.265, #10% accepted error
#     "p_preterm_l23": 0.035,      #5% accepted error
#     "p_preterm_l45": 0.209,      #5% accepted error
#     "p_cs_l23": 0.024,           #5% accepted error
#
#     "p_cs_l45": 0.264,           #5% accepted error
#     "p_unnecessary_cs": 0.67,    #10% accepted error
#     "p_l23_post": 0.27,          #5% accepted error
#     "p_l45_post": 0.38,          #5% accepted error
#     "p_mcr_l23_l45": 0.2,        #10% accepted error
#
#     "p_SMO_fac": 0.0375,         #10% accepted error
#     "p_pph_l45": 0.086,          #5% accepted error
#     "p_mat_sepsis_l45": 0.15,    #5% accepted error
#     "p_eclampsia_fac": 0.02,     #5% accepted error
#     "p_obstructed_fac": 0.045,   #5% accepted error
# }
target_weights = {
    # Key: (target_value, accepted_error_percentage)
    "p_elcs_fac": (0.066, 5),
    "p_elcs_preterm_l45": (0.265, 10),
    "p_preterm_l23": (0.035, 5),
    "p_preterm_l45": (0.209, 5),
    "p_cs_l23": (0.024, 5),
    "p_cs_l45": (0.264, 5),
    "p_unnecessary_cs": (0.67, 10),
    "p_l23_post": (0.27, 5),
    "p_l45_post": (0.38, 5),
    "p_mcr_l23_l45": (0.2, 10),
    "p_SMO_fac": (0.0375, 10),
    "p_pph_l45": (0.086, 5),
    "p_mat_sepsis_l45": (0.15, 5),
    "p_eclampsia_fac": (0.02, 5),
    "p_obstructed_fac": (0.045, 5)
}

# Generate unique seeds for reproducibility
total_runs = 30
seeds = np.random.default_rng(2025).integers(low=0, high=1e6, size=total_runs)

# Objective function for Bayesian Optimization
@use_named_args(param_space)
def objective(**params):

    from parameters import get_parameters, get_slider_params
    from model_run import run_model_dash
    from global_func import reset_flags, reset_E, reset_HSS, reset_S

    MODEL = {
        "int_period": 36,
        "n_months": 36,
    }

    slider_params = get_slider_params()
    results = []

    # Calculate max allowed delta to prevent severe > 90
    max_delta = 90 - params['t_l23_l45_notsevere']
    params['delta_severe_vs_notsevere'] = min(
        params['delta_severe_vs_notsevere'],  # Original delta
        max_delta  # Ensures severe ≤ 90
    )
    params['t_l23_l45_severe'] = params['t_l23_l45_notsevere'] + params['delta_severe_vs_notsevere']

    for run in range(total_runs):
        base_seed = seeds[run]
        rng_param = np.random.default_rng(base_seed)
        b_param = get_parameters(rng=rng_param)
        b_flags = reset_flags()
        b_HSS = reset_HSS(slider_params)
        b_S = reset_S(slider_params)
        b_E = reset_E()
        b_param.update({"E": b_E, "S": b_S, "HSS": b_HSS})

        # Replace parameters with those to calibrate
        b_param['p_elec_CS|highrisk'] = params['p_elec_CS|highrisk']
        b_param['p_elec_CS|preterm'] = params['p_elec_CS|preterm']
        b_param['t_l23_l45_preterm'] = params['t_l23_l45_preterm']
        b_param['p_cs_capacity'][1] = params['p_cs_capacity_l23']
        b_param['p_cs_capacity'][2] = params['p_cs_capacity_l45']

        b_param['p_cs_capacity'][3] = params['p_cs_capacity_l45']
        b_param['sen_comp_trad'] = params['sen_comp_trad']
        b_param['spec_comp_trad'] = params['spec_comp_trad']
        b_param['t_l23_l45_notsevere'] = params['t_l23_l45_notsevere']
        b_param['t_l23_l45_severe'] = params['t_l23_l45_severe']

        b_param['p_eclampsia'] = params['p_eclampsia']
        b_param['p_OL_scale'] = params['p_OL_scale']
        b_param['p_pph_other'] = params['p_pph_other']
        b_param['p_mat_sepsis_other'] = params['p_mat_sepsis_other']
        b_param['p_comp_severe_lowrisk'] = params['p_comp_severe_lowrisk']

        #update parameters
        b_param = calculate_derived_parameters(b_param)

        #run the model
        n_months = MODEL["n_months"]
        int_period = MODEL["int_period"]
        _, outcomes = run_model_dash(b_param, b_flags, n_months, int_period, base_seed=base_seed)

        # Calculate outcomes
        outcomes['i_CS'] = np.where(outcomes['i_mod'].isin(["EmCS", "ELCS"]), 1, 0)
        outcomes['i_loc_grouped'] = np.where(outcomes['i_loc_new_v2'] == 0, 0,
                                             np.where(outcomes['i_loc_new_v2'] == 1, 1, 2))
        outcomes['i_facility'] = np.where(outcomes['i_loc_new_v2'] > 0, 1, 0)
        outcomes['i_home'] = np.where(outcomes['i_loc_new_v2'] == 0, 1, 0)
        mcr_l23_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 1]['i_comp_death_new'].mean(), nan=0.0)
        mcr_l45_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 2]['i_comp_death_new'].mean(), nan=0.0)
        p_mcr_l23_l45 = mcr_l23_mean / mcr_l45_mean if mcr_l45_mean != 0 else 0


        results.append({
            "p_elcs_fac": outcomes.groupby('i_facility')['i_elec_CS'].mean().get(1, 0),
            "p_elcs_preterm_l45": np.nan_to_num(
                outcomes[(outcomes['i_loc_grouped'] == 2) & (outcomes['i_preterm'] == 1)]['i_elec_CS'].mean(), nan=0.0),
            "p_preterm_l23": np.nan_to_num(
                outcomes[(outcomes['i_loc_grouped'] == 1)]['i_preterm'].mean(), nan=0.0),
            "p_preterm_l45": np.nan_to_num(
                outcomes[(outcomes['i_loc_grouped'] == 2)]['i_preterm'].mean(), nan=0.0),
            "p_cs_l23": outcomes.groupby('i_loc_grouped')['i_CS'].mean().get(1, 0),

            "p_cs_l45": outcomes.groupby('i_loc_grouped')['i_CS'].mean().get(2, 0),
            "p_unnecessary_cs": np.nan_to_num(
                outcomes[outcomes['i_mod'] == "EmCS"]['i_unnecessary_cs'].mean(), nan=0.0),
            "p_l23_post": np.nan_to_num((outcomes['i_loc_grouped'] == 1).mean(), nan=0.0),
            "p_l45_post": np.nan_to_num((outcomes['i_loc_grouped'] == 2).mean(), nan=0.0),
            "p_mcr_l23_l45": p_mcr_l23_l45,

            "p_SMO_fac": np.nan_to_num(
                outcomes[(outcomes['i_severe_new'] == 1) & (outcomes['i_facility'] == 1)]['i_mat_death'].mean(), nan=0.0),
            "p_pph_l45": outcomes.groupby('i_loc_grouped')['i_pph_new'].mean().get(2, 0),
            "p_mat_sepsis_l45": outcomes.groupby('i_loc_grouped')['i_mat_sepsis'].mean().get(2, 0),
            "p_eclampsia_fac": outcomes.groupby('i_facility')['i_eclampsia'].mean().get(1, 0),
            "p_obstructed_fac": outcomes.groupby('i_facility')['i_OL_final'].mean().get(1, 0),
        })


    # Convert to DataFrame and compute mean
    df_results = pd.DataFrame(results).mean().to_dict()

    # Add this line here for early stopping access
    objective.last_df_results = df_results

    # Compute relative RMSE
    # relative_squared_errors = [ ((df_results[k] - target_values[k]) / target_values[k])**2 for k in target_values ]
    # rmse = math.sqrt(sum(relative_squared_errors) / len(relative_squared_errors))
    rmse = weighted_rmse(df_results, target_weights)

    # Print current evaluation
    print("\n--- Calibration Iteration ---")
    print("Parameters:")
    for k, v in params.items():
        print(f"  {k:22s}: {v:.4f}")

    # print("\nTarget vs Simulated Outcomes:")
    # for k in target_values:
    #     sim = df_results.get(k, 0)
    #     target = target_values[k]
    #     error = sim - target
    #     rel_error = error / target
    #     print(f"  {k:22s} | Simulated: {sim:.4f}, Target: {target:.4f}, Relative Error: {rel_error:+.2%}")
    # print(f"\nRMSE Loss: {rmse:.6f}")
    print("\nError Report (% of allowed tolerance):")
    for metric, (target, pct) in target_weights.items():
        simulated = df_results[metric]
        error_pct = 100 * (simulated - target) / target  # Actual error percentage
        tolerance_pct = pct  # Your predefined accepted error
        normalized = error_pct / tolerance_pct  # How much of allowed tolerance is used

        print(f"  {metric:20s}: {target:.4f} → {simulated:.4f} "
              f"({error_pct:+.1f}% / {tolerance_pct}% = {normalized:.2f}x allowed)")

    print(f"\nWeighted RMSE: {rmse:.3f} (Target: <1.0)")
    print("-----------------------------\n")
    return rmse

In [3]:
# Track RMSE history
rmse_history = []

class EarlyStopper:
    def __init__(self, threshold_rmse=1.0, max_patience=10):
        """
        Args:
            threshold_rmse: Target weighted RMSE (default 1.0 = all errors within tolerances)
            max_patience: How many iterations to wait without improvement before stopping
        """
        self.threshold_rmse = threshold_rmse
        self.max_patience = max_patience
        self.best_loss = float('inf')
        self.best_params = None
        self.best_df_results = None
        self.patience_counter = 0
        self.converged = False

    def check_tolerance_violations(self, sim_results):
        """Returns list of metrics exceeding their allowed tolerances"""
        violations = []
        for metric, (target, accepted_pct) in target_weights.items():
            error_pct = 100 * abs(sim_results.get(metric, 0) - target) / target
            if error_pct > accepted_pct:
                violations.append(f"{metric} ({error_pct:.1f}% > {accepted_pct}%)")
        return violations

    def __call__(self, *args, **kwargs):
        loss = objective(*args, **kwargs)  # This now returns weighted RMSE

        # Track best results
        if loss < self.best_loss:
            self.best_loss = loss
            self.best_params = args[0] if args else None
            self.best_df_results = getattr(objective, "last_df_results", {})
            self.patience_counter = 0  # Reset patience on improvement
        else:
            self.patience_counter += 1

        # Check stopping conditions
        if self.best_loss <= self.threshold_rmse:
            violations = self.check_tolerance_violations(self.best_df_results)
            if not violations:
                print(f"\n✅ Early stopping: All targets within tolerances (RMSE={self.best_loss:.3f} ≤ {self.threshold_rmse})")
                self.converged = True
                raise StopIteration
            else:
                print(f"\n⚠️ RMSE threshold met but tolerances violated: {', '.join(violations)}")

        if self.patience_counter >= self.max_patience:
            print(f"\n⏹️ Early stopping: No improvement for {self.max_patience} iterations (Best RMSE={self.best_loss:.3f})")
            raise StopIteration

        return loss

In [4]:
def save_results(best_params, best_loss, rmse_history, param_space, target_weights):
    """
    Save optimization results with enhanced reporting.

    Args:
        best_params: Optimized parameters
        best_loss: Best weighted RMSE achieved
        rmse_history: List of RMSE values over iterations
        param_space: Parameter search space
        target_weights: Dictionary of target tolerances
    """
    # Save best parameters
    param_names = [dim.name for dim in param_space]
    best_param_dict = dict(zip(param_names, best_params))
    best_param_dict["Weighted_RMSE"] = best_loss
    df_best = pd.DataFrame([best_param_dict])
    df_best.to_csv("best_parameters_labor_delivery.csv", index=False)

    # Save full optimization history
    history_df = pd.DataFrame({
        "Iteration": range(1, len(rmse_history)+1),
        "Weighted_RMSE": rmse_history
    })
    history_df.to_csv("rmse_history_labor_delivery.csv", index=False)

    # Enhanced plot with convergence threshold
    plt.figure(figsize=(12, 6))
    plt.plot(history_df["Iteration"], history_df["Weighted_RMSE"],
             marker='o', linestyle='-', label='Weighted RMSE')
    plt.axhline(y=1.0, color='r', linestyle='--',
                label='Tolerance Threshold (RMSE=1.0)')
    plt.xlabel("Iteration")
    plt.ylabel("Weighted RMSE")
    plt.title(f"Optimization Progress (Final RMSE: {best_loss:.3f})")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("optimization_progress_labor_delivery.png", dpi=300)
    plt.close()

    # Save target achievement report
    if hasattr(objective, "last_df_results"):
        target_report = []
        for metric, (target, pct) in target_weights.items():
            simulated = objective.last_df_results.get(metric, 0)
            error_pct = 100 * (simulated - target) / target
            target_report.append({
                "Metric": metric,
                "Target": target,
                "Simulated": simulated,
                "Error%": error_pct,
                "Tolerance%": pct,
                "Within_Tolerance": abs(error_pct) <= pct
            })

        pd.DataFrame(target_report).to_csv("target_achievement_report_labor_delivery.csv", index=False)

def optimize_model():
    """
    Run Bayesian optimization with improved early stopping.
    Returns either optimization result or None if stopped early.
    """
    # Initialize with weighted RMSE threshold
    early_stopper = EarlyStopper(
        threshold_rmse=1.0,     # All errors within accepted tolerances
        max_patience=1000       # Stop if no improvement for 20 iterations
    )

    global rmse_history
    rmse_history = []

    try:
        result = gp_minimize(
            func=early_stopper,
            dimensions=param_space,
            n_calls=1000,
            random_state=42,
            n_random_starts=150,
            verbose=True,
            n_jobs=-1  # Parallel execution
        )

        if early_stopper.converged:
            print("\n✅ Optimization converged successfully!")
        else:
            print("\n⚠️ Optimization completed but did not fully converge")

        print(f"Best Weighted RMSE: {result.fun:.3f}")
        save_results(result.x, result.fun, rmse_history, param_space, target_weights)
        return result

    except StopIteration:
        print("\n⏹️ Optimization stopped by early stopping criteria")
        print(f"Best Weighted RMSE achieved: {early_stopper.best_loss:.3f}")

        if early_stopper.best_params is not None:
            print("\nBest Parameters:")
            for name, val in zip([d.name for d in param_space], early_stopper.best_params):
                print(f"  {name:22s}: {val:.4f}")

            save_results(
                early_stopper.best_params,
                early_stopper.best_loss,
                rmse_history,
                param_space,
                target_weights
            )

        return None

# Run the optimization
if __name__ == "__main__":
    final_result = optimize_model()

    # Additional post-optimization analysis
    if final_result is not None or hasattr(objective, "last_df_results"):
        print("\nTarget Achievement Summary:")
        for metric, (target, pct) in target_weights.items():
            simulated = objective.last_df_results.get(metric, 0)
            status = "✓" if abs(100*(simulated-target)/target) <= pct else "✗"
            print(f"  {status} {metric:20s}: {simulated:.4f} (Target: {target:.4f} ±{pct}%)")

Iteration No: 1 started. Evaluating function at random point.

--- Calibration Iteration ---
Parameters:
  p_elec_CS|highrisk    : 0.1613
  p_elec_CS|preterm     : 0.5550
  t_l23_l45_preterm     : 95.5938
  p_cs_capacity_l23     : 0.0637
  p_cs_capacity_l45     : 0.1446
  sen_comp_trad         : 0.5240
  spec_comp_trad        : 0.6240
  t_l23_l45_notsevere   : 63.3483
  delta_severe_vs_notsevere: 6.5718
  p_comp_severe_lowrisk : 0.0686
  p_eclampsia           : 0.0111
  p_OL_scale            : 0.9166
  p_pph_other           : 0.0194
  p_mat_sepsis_other    : 0.0100
  t_l23_l45_severe      : 69.9202

Error Report (% of allowed tolerance):
  p_elcs_fac          : 0.0660 → 0.0737 (+11.6% / 5% = 2.32x allowed)
  p_elcs_preterm_l45  : 0.2650 → 0.1866 (-29.6% / 10% = -2.96x allowed)
  p_preterm_l23       : 0.0350 → 0.0136 (-61.0% / 5% = -12.21x allowed)
  p_preterm_l45       : 0.2090 → 0.2120 (+1.4% / 5% = 0.28x allowed)
  p_cs_l23            : 0.0240 → 0.0318 (+32.7% / 5% = 6.53x allowed)
 

KeyboardInterrupt: 

In [ ]:
# class EarlyStopper:
#     def __init__(self, threshold_rmse=0.05, relative_threshold=0.05):
#         self.threshold_rmse = threshold_rmse
#         self.relative_threshold = relative_threshold
#         self.best_loss = float('inf')
#         self.call_count = 0
#         self.best_params = None
#         self.best_df_results = None
#
#     def meets_relative_criteria(self, sim_results):
#         for k in target_values:
#             target = target_values[k]
#             sim = sim_results.get(k, 0)
#             rel_error = abs(sim - target) / target
#             if rel_error > self.relative_threshold:
#                 return False
#         return True
#
#     def __call__(self, *args, **kwargs):
#         loss = objective(*args, **kwargs)
#         rmse_history.append(loss)
#
#         self.call_count += 1
#         if loss < self.best_loss:
#             self.best_loss = loss
#             self.best_params = args[0] if args else None
#
#         sim_results = getattr(objective, "last_df_results", {})
#         if self.best_loss < self.threshold_rmse and self.meets_relative_criteria(sim_results):
#             print(f"\n⏹️ Early stopping triggered. RMSE: {self.best_loss:.6f}")
#             raise StopIteration
#
#         return loss

In [ ]:
# def save_results(best_params, best_loss, rmse_history, param_space):
#     param_names = [dim.name for dim in param_space]
#     best_param_dict = dict(zip(param_names, best_params))
#     best_param_dict["Best RMSE"] = best_loss
#     df_best = pd.DataFrame([best_param_dict])
#     df_best.to_csv("best_parameters_labor_delivery.csv", index=False)
#
#     pd.DataFrame({"Iteration": range(1, len(rmse_history)+1), "RMSE": rmse_history}).to_csv("rmse_history_labor_delivery.csv", index=False)
#
#     plt.figure(figsize=(10, 6))
#     plt.plot(range(1, len(rmse_history)+1), rmse_history, marker='o')
#     plt.xlabel("Iteration")
#     plt.ylabel("RMSE Loss")
#     plt.title("RMSE Loss Over Iterations")
#     plt.grid(True)
#     plt.tight_layout()
#     plt.savefig("rmse_plot_labor_delivery.png")
#     plt.close()

In [ ]:
# def optimize_model():
#     early_stopper = EarlyStopper(threshold_rmse=0.05, relative_threshold=0.05)
#     try:
#         result = gp_minimize(
#             func=early_stopper,
#             dimensions=param_space,
#             n_calls=10,
#             random_state=42,
#             n_random_starts=2,
#             base_estimator='ET',
#             verbose=True,
#             n_jobs=-1,
#         )
#         print("\n✅ Optimization completed.")
#         print("Best Parameters:", result.x)
#         print("Best Loss:", result.fun)
#         save_results(result.x, result.fun, rmse_history, param_space)
#         return result
#
#     except StopIteration:
#         print("\n✅ Optimization stopped early.")
#         print(f"Best RMSE achieved: {early_stopper.best_loss:.6f}")
#         if early_stopper.best_params is not None:
#             print("Best Parameters:")
#             for name, val in zip([d.name for d in param_space], early_stopper.best_params):
#                 print(f"  {name:22s}: {val:.4f}")
#             save_results(early_stopper.best_params, early_stopper.best_loss, rmse_history, param_space)
#         return None
#
# # Run the optimization
# if __name__ == "__main__":
#     optimize_model()